In [33]:
import numpy as np
import scipy
from tueplots import bundles
import matplotlib.pyplot as plt
## setting matplotlib context
from cycler import cycler
from matplotlib.cm import get_cmap
cmap = get_cmap("tab10",8)
palette = [cmap(i) for i in range(8)]
rc = bundles.neurips2024(usetex=False)
rc.update({
    # Set the line/bar color cycle (this is what affects ax.plot)
    "axes.prop_cycle": cycler(color=palette),
    # Optional readability tweaks
    "legend.frameon": False,
    "axes.grid": False,
})
np.random.seed(0)


/var/folders/cx/cy7pq56x7m32_zfmgnslz8cnlnlryt/T/ipykernel_72034/2843679705.py:8: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed in 3.11. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap()`` or ``pyplot.get_cmap()`` instead.
  cmap = get_cmap("tab10",8)


### Outline:

1. Using a finite superset D_shadow, train M shadow models.
2. Compute P_in and P_out per sample in the superset from these shadow models.
   
[TRAIN PHASE]

Train M_test models on subsets of D_shadow and evaluate TPR/FPR PX for them using the P_in and P_out.

[TEST PHASE]

1. For each sample in the superset, build new sets (with M_test) of D_in/ D_out based on LOO with remaining samples drawn fresh from population.
2. Re-compute IN/ OUT scores for each sample.
3. Use P_in and P_out computed from shadow models trained on superset to compute average TPR/FPR PX on the M_test models.

In [34]:
np.random.seed(0)

N = 1000
N_train = 500
sigma = 1.0
alpha = 0.01
eps = 1e-12
M = 10_000
M_test = 1_000

# Target Superset #
X = scipy.stats.norm.rvs(loc=0.0, scale=sigma, size=N)

# Membership Matrix #
memberships = np.zeros((M + M_test, N), dtype=bool)
for r in range(M + M_test):
    idx = np.random.choice(N, size=N_train, replace=False)
    memberships[r, idx] = True

# Computing Model Statistics #
model_means = (memberships @ X) / N_train   # (M+M_test,)
model_means, X = np.expand_dims(model_means,axis=1), np.expand_dims(X,axis=1)
stats      = model_means @ X.T             # (M+M_test, N)


# Splitting the test models trained from superset and shadow models for stats #
shadow_test_stats, shadow_test_indices = stats[:M_test,:], memberships[:M_test,:]
shadow_stats, shadow_indices = stats[M_test:,:], memberships[M_test:,:]

In [35]:
mu_ins, mu_outs ,sigma_ins, sigma_outs = np.zeros(N), np.zeros(N), np.zeros(N), np.zeros(N)
for n in range(N):
    curr_stats, curr_indices = shadow_stats[:,n], shadow_indices[:,n]
    in_stats, out_stats = curr_stats[curr_indices], curr_stats[~curr_indices]
    mu_ins[n], mu_outs[n], sigma_ins[n], sigma_outs[n] = np.mean(in_stats), np.mean(out_stats), np.std(in_stats), np.std(out_stats)

In [36]:
def compute_llr_grid(
    stats:     np.ndarray,  # (K, N)
    mu_ins:    np.ndarray,  # (N,)
    mu_outs:   np.ndarray,  # (N,)
    sigma_ins:  np.ndarray, # (N,)
    sigma_outs: np.ndarray, # (N,)
    eps: float = 1e-8,
) -> np.ndarray:            # (K, N)
    return (
        np.log(sigma_ins + eps) - np.log(sigma_outs + eps)
        + 0.5 * (stats - mu_outs) ** 2 / (sigma_outs + eps) ** 2
        - 0.5 * (stats - mu_ins)  ** 2 / (sigma_ins  + eps) ** 2
    )

shadow_test_llrs = compute_llr_grid(shadow_test_stats, mu_ins, mu_outs, sigma_ins, sigma_outs)

tau_px = np.nanquantile(np.where(~shadow_test_indices, shadow_test_llrs, np.nan), 1 - alpha, axis=0)
tpr_px_shadow = (((shadow_test_llrs > tau_px) & shadow_test_indices).sum(axis=0)
                / shadow_test_indices.sum(axis=0))
fpr_px_shadow = (((shadow_test_llrs > tau_px) & ~shadow_test_indices).sum(axis=0)
                / (~shadow_test_indices).sum(axis=0))
print(tpr_px_shadow.mean(),fpr_px_shadow.mean())

0.013436111957434013 0.010895926783256984


In [37]:
stats_test_in, stats_test_out = np.zeros((M_test//2,N)), np.zeros((M_test//2,N))
for n in range(N):
    # population-based test sets
    # for a single element, assume we have M_test (IN + OUT) test models build on LOO principle.
    X_test_out = scipy.stats.norm.rvs(loc=0.0, scale=sigma, size=(M_test//2,N_train))
    X_test_in = np.column_stack([X_test_out[:,1:], np.ones(M_test//2) * X[n]])
    in_sums, out_sums = X_test_in.mean(axis=1), X_test_out.mean(axis=1)
    stats_test_in[:,n] = np.expand_dims(in_sums,axis=1) @ np.expand_dims(X[n],axis=1)[:,0]
    stats_test_out[:,n] = np.expand_dims(out_sums,axis=1) @ np.expand_dims(X[n],axis=1)[:,0]

pop_test_llrs_in = compute_llr_grid(stats_test_in, mu_ins, mu_outs, sigma_ins, sigma_outs)
pop_test_llrs_out = compute_llr_grid(stats_test_out, mu_ins, mu_outs, sigma_ins, sigma_outs)
tau_px_test = np.quantile(pop_test_llrs_out, q=1-alpha,axis=0)
tpr_px_test = (pop_test_llrs_in > tau_px_test).mean(axis=0)
fpr_px_test = (pop_test_llrs_out > tau_px_test).mean(axis=0)

print(tpr_px_test.mean(), fpr_px_test.mean())

0.012188000000000003 0.01
